In [ ]:
# Needed::

import os
import json
import torch
import random
import numpy as np
import torchvision
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from PIL import Image, ImageFile
import matplotlib.patches as patches
from torchvision import transforms, models
# from torchvision.references.detection import engine
# from engine import train_one_epoch, evaluate
from torchvision.datasets import CocoDetection
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import FasterRCNN
from torchvision.transforms import v2 as transforms
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

In [ ]:
import os, json, random
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from PIL import Image
from sklearn.metrics import classification_report
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Classes (multi-label)
CLASSES = ['mat_bo_phan', 'rach', 'mop_lom', 'tray_son', 'thung', 'vo_kinh', 'be_den']
CLASS_TO_IDX = {name: i for i, name in enumerate(CLASSES)}
BATCH_SIZE = 32
LR = 0.0001
EPOCHS = 10
NUM_WORKERS = 4

In [ ]:
class VehicleDamageDataset(Dataset):
    def __init__(self, img_dir, annotations_file, transform=None):
        self.img_dir = img_dir
        self.transform = transform
        with open(annotations_file) as f:
            self.annotations = json.load(f)
        self.valid_images = [img for img in self.annotations if os.path.exists(os.path.join(img_dir, img))]

    def __len__(self):
        return len(self.valid_images)

    def __getitem__(self, idx):
        img_name = self.valid_images[idx]
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert('RGB')

        target = torch.zeros(len(CLASSES), dtype=torch.float32)
        for region in self.annotations[img_name].get('regions', []):
            class_name = region['class']
            if class_name in CLASS_TO_IDX:
                target[CLASS_TO_IDX[class_name]] = 1.0

        if self.transform:
            image = self.transform(image)
        return image, target

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_data = VehicleDamageDataset(
    '/kaggle/input/vehide-dataset-automatic-vehicle-damage-detection/image/image',
    '/kaggle/input/vehide-dataset-automatic-vehicle-damage-detection/0Train_via_annos.json',
    transform
)

val_data = VehicleDamageDataset(
    '/kaggle/input/vehide-dataset-automatic-vehicle-damage-detection/validation/validation',
    '/kaggle/input/vehide-dataset-automatic-vehicle-damage-detection/0Val_via_annos.json',
    transform
)

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, num_workers=4)

In [ ]:
class DamageNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.base = models.resnet50(weights='DEFAULT')
        in_features = self.base.fc.in_features
        self.base.fc = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.base(x)

model = DamageNet(len(CLASSES)).to(device)

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=20, verbose=True)

best_val_loss = float('inf')
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    val_loss = 0.0
    correct = 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            val_loss += criterion(outputs, labels).item()

            preds = (torch.sigmoid(outputs) > 0.5).int()
            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

            correct += (preds == labels.bool()).all(dim=1).sum().item()

    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)

    train_loss /= len(train_loader)
    val_loss /= len(val_loader)
    val_acc = correct / len(val_data)
    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'model_state_dict': model.state_dict(),
            'class_to_idx': CLASS_TO_IDX,
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
        }, '/kaggle/working/best_damage_model.pth')

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    print("\n--- Classification Report ---")
    print(classification_report(all_labels, all_preds, target_names=CLASSES, zero_division=0))

print("Training complete!")